# v16 — fully verified T4 candidate

115 kernel tests + 79 Soup tests and **10/10 checks passed** in two paired runs.
85.07 → 60.41 seconds; 14345 → 5601 MiB; eval loss +0.001%.
Exact measured code and raw reports were hash-verified locally.

Select **GPU T4 x2**, **Internet On**, then **Run All**. Keep COMMIT pinned.
A unique checkout preserves old results. The branch tip may contain experiments.


In [ ]:
REPO_URL = "https://github.com/promaladoi/Decentrathon-6.0.git"
COMMIT = "4a5cfd20fd79e597e15919f3f19f959a6b4d689a"
VERSION = "v16"


In [ ]:
import subprocess, os, json, zipfile, uuid
from pathlib import Path
from IPython.display import display, FileLink
print(subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],text=True))
subprocess.run(['python','-m','pip','install','-q','soup-cli[train]==0.77.0','transformers==5.16.1','peft==0.20.0','trl==0.29.1','bitsandbytes==0.50.2','accelerate==1.14.0','pytest'],check=True)


In [ ]:
# Export the existing v13 run if it is still available, without retraining.
old=Path('/kaggle/working/nurzhigit_v13')
old_out=old/'runs_codex_v13'
if all((old_out/n).exists() for n in ('result.json','checks.json')):
    with zipfile.ZipFile('/kaggle/working/codex_v13.zip','w',zipfile.ZIP_DEFLATED) as z:
        for n in ('result.json','checks.json'): z.write(old_out/n,n)
        for f in (old/'solution').rglob('*.py'): z.write(f,str(f.relative_to(old)))
        for n in ('adapter_config.json','adapter_model.safetensors'):
            f=old_out/'solution_1/adapter'/n
            if f.exists(): z.write(f,'adapter/'+n)
    display(FileLink('codex_v13.zip'))
else:
    print('No previous v13 artifacts in this session; continuing with the pinned experiment.')


In [ ]:
assert not subprocess.check_output(['nvidia-smi','--query-compute-apps=pid','--format=csv,noheader'],text=True).strip(), 'GPU is busy: stop the other GPU work before running this notebook.'
p=Path('/kaggle/working')/('nurzhigit_'+VERSION+'_'+uuid.uuid4().hex[:8])
subprocess.run(['git','clone','-q',REPO_URL,str(p)],check=True,timeout=120)
subprocess.run(['git','-C',str(p),'checkout','-q',COMMIT],check=True)
print('MEASURED CODE:',subprocess.check_output(['git','-C',str(p),'rev-parse','HEAD'],text=True).strip())
subprocess.run(['python','task/prepare_data.py'],cwd=p,check=True)


In [ ]:
completed = subprocess.run(['python','04_submission/run_candidate.py','--version',VERSION],cwd=p)
display(FileLink('codex_'+VERSION+'.zip'))
assert completed.returncode == 0, 'Experiment stopped or rejected; inspect its output/archive.'
